In [ ]:
# Imports
import cProfile
import pstats
import matplotlib.pyplot as plt
from regions import Regions
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import WCSAxes
from astropy.coordinates import SkyCoord, FK5
from spectral_cube import SpectralCube
from velocity_tools import extract_streamline, gradient_descent, stream_lines_grad
from velocity_tools import stream_lines # won't use this directly, but needed to compare with stream_lines_grad
import os
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import value_and_grad
import pandas as pd
import optax

import warnings
warnings.filterwarnings('ignore', message='.*PV2_1.*')
warnings.filterwarnings('ignore', message='.*PV2_2.*')
warnings.filterwarnings('ignore', message='.*TIMESYS.*')

# Settings
hltau_c= SkyCoord("4h31m38.43s", "+18d13m57.19s", frame='fk5')
hltau_ref = hltau_c.skyoffset_frame()
iras2a_c = SkyCoord("3h28m55.569s", "+31d14m37.025s", frame='fk5')
iras2a_ref = iras2a_c.skyoffset_frame()
b5irs1_c = SkyCoord("3h47m41.577s", "+32d51m43.745s", frame='fk5')
b5irs1_ref = b5irs1_c.skyoffset_frame()
distance_hltau = 147 #parsecs
distance_iras2a = 293 #parsecs
distance_b5irs1 = 302 #parsecs
# choose which distance
distance = distance_b5irs1

# cubefile = 'test_data/HLTau/HLTAU_HCOp32.fits'
# file_Tpeak = 'test_data/HLTau/HLTAU_HCOp32_Tpeak.fits'
cubefile = 'test_data/B5IRS1/B5-IRS1_CD_c-HCCCH_6_0_6_5_1_5_sc_contsub_sm-merged-pbcor.fits'

# some constants
G = 6.67430e-11 * (1e-3)**2 * (1.988416e30) / (1.4959787e11) # in au (km/s)^2 * Msol^-1
au_in_km = 1.4959787e8 #km



### Original cube with spectra: Prepare the 1D streamer emission from the cube

In [ ]:
# get the spectralcube object from the data using spectral-cube
hdu = fits.open(cubefile)[0]
cube = SpectralCube.read(hdu).with_spectral_unit(u.km/u.s, rest_value=hdu.header['RESTFREQ']*u.Hz)

# # mask the cube using the region file
region_file = 'test_data/B5IRS1/streamer_region_small.reg'
regions = Regions.read(region_file, format='ds9')
# # create spatial mask from first region
reg = regions[0]
# # make 2D mask in celestial plane
mask2d = reg.to_pixel(cube.wcs.celestial).to_mask()
# # convert to image-sized boolean array
spatial_mask = mask2d.to_image(cube.shape[1:]).astype(bool)
# # expand to 3D for spectral cube
mask3d = np.broadcast_to(spatial_mask, cube.shape)
# # apply mask
cube = cube.with_mask(mask3d)


# TODO: this streamer extraction should be replaced with clustering-based streamer extraction

# extract the subcube with the streamer (we got this from the tipsy tutorial, no need to plot)
## Limits for extracting subcube with streamer

'''
vmin = 7    # Min. vel. of streamer
vmax = 10
xmin = -3   # Min R.A. offset (arcsec) to consider for streamer
xmax = -1
ymin = -3   # Min. Decl. offset (arcsec) to consider for streamer
ymax = 0.5
rms_thresh = 4  # sigma threshold for streamer
'''

vmin = 8
vmax = 11
xmin = -10
xmax =10
ymin = -10
ymax = 10
rms_thresh = 5

# extract the streamer subcube
info_header = cube.header  # header of the cube, contains required information
x_conv_fac = 1/60/60/info_header['CDELT1']
xmin_p = int((xmin*x_conv_fac)+info_header['CRPIX1'])
xmax_p = int((xmax*x_conv_fac)+info_header['CRPIX1'])
y_conv_fac = 1/60/60/info_header['CDELT2']
ymin_p = int((ymin*y_conv_fac)+info_header['CRPIX2'])
ymax_p = int((ymax*y_conv_fac)+info_header['CRPIX2'])
vunit = cube.spectral_axis.unit
## Note: a check can be added to see if requested limits are within the limits of the cube itself

streamer_cubev = cube.spectral_slab(vmin*vunit,vmax*vunit)    # Selecting velocities
streamer_cubevc = streamer_cubev[:,min(ymin_p,ymax_p):max(ymin_p,ymax_p)   
                    ,min(xmin_p,xmax_p):max(xmin_p,xmax_p)]   # Selecting pixels
#     print(min(ymin_p,ymax_p),max(ymin_p,ymax_p),min(xmin_p,xmax_p),max(xmin_p,xmax_p)) 
streamer_cube = streamer_cubevc.with_mask(streamer_cubevc > rms_thresh*streamer_cubevc.mad_std())  # Removing low flux values 


In [ ]:
from velocity_tools.streamfit.extract_streamline import cartesian_to_polar, get_distance_metric


n_points = 10 # the number of points we want to reduce the data to


# try a method binning by polar angle instead
def get_polar_angle_metric(ra_coords, dec_coords):

    pc_r, pc_theta = cartesian_to_polar(ra_coords, dec_coords)
    polar_angle_metric = pc_theta
    return polar_angle_metric

def reduce_to_1D_polar(streamer_cube, n_elements=10):
    '''
    This function will reduce a cube of emission to a 1D 'streamline', 
    by weighted means in polar angle bins.

    In this function, 'pc' is short for point cloud.

    Parameters
    ----------
    streamer_cube : SpectralCube object, should contain only streamer emission
    n_elements : int, number of elements to reduce the cube to


    Returns
    -------
    pc_means : array of shape (3, n_elements), the weighted mean coordinates of each bin
    index 0 = RA offsets (arcsec)
    index 1 = Dec offsets (arcsec)
    index 2 = velocity (km/s)
    '''
    print('Starting reduction')
    nz, ny, nx = streamer_cube.shape

    # create coordinate arrays for RA and Dec in arcsec
    y_indices, x_indices = np.mgrid[0:ny, 0:nx]
    world_coords = streamer_cube.wcs.celestial.pixel_to_world_values(x_indices.ravel(), y_indices.ravel())
    ra_coords = (world_coords[0].reshape(ny, nx) - streamer_cube.header['CRVAL1']) * 60 * 60
    ra_coords = ra_coords * np.cos(streamer_cube.header['CRVAL2'] * np.pi / 180) # cos(dec) correct for declination. in arcsec
    dec_coords = (world_coords[1].reshape(ny, nx) - streamer_cube.header['CRVAL2']) * 60 * 60 # in arcsec

    # create velocity array in km/s
    #TODO: fix this to use spectral_axis and WCS instead of header keywords, to be more robust
    v_coords = streamer_cube.spectral_axis.to(u.km/u.s).value - (streamer_cube.header['CRVAL3']*1e-3)

    print('Created coordinate arrays')

    # get data and mask
    pcloud = np.array(streamer_cube)
    rms_mask = ~np.isnan(pcloud)
    flux = pcloud[rms_mask]

    # get indices of valid points in pc
    pc_indices = np.indices(pcloud.shape) # indices of all points in pc
    pc_z = pc_indices[0][rms_mask] # z indices of points in pc
    pc_y = pc_indices[1][rms_mask] # y indices of points in pc
    pc_x = pc_indices[2][rms_mask] # x indices of points in pc

    print('Got point cloud with', len(flux), 'points')

    # extract coordinates of valid points using the arrays above
    pc_ra = ra_coords[pc_y, pc_x]
    pc_dec = dec_coords[pc_y, pc_x]
    pc_v = v_coords[pc_z]
    pc_coords = np.array([pc_ra, pc_dec, pc_v]) # shape (3, n_points)   

    # compute polar angle metric to bin the point cloud
    polar_angle_metric = get_polar_angle_metric(pc_coords[0], pc_coords[1])
    b_per = np.linspace(0, 100, n_elements+1) # percentiles to bin the pc into
    partitions = np.array([np.percentile(polar_angle_metric, per) for per in b_per])

    print("Partition boundaries for polar angle metric:", np.round(partitions, 3))

    # take flux-weighted means and stds in each bin
    pc_means = np.zeros((3, n_elements))
    pc_stds = np.zeros((3, n_elements))
    for i in range(n_elements):
        # identify points in this bin, add weighted means and weighted stds
        angle_indices = (polar_angle_metric > partitions[i]) & (polar_angle_metric <= partitions[i+1])
        pc_means[:, i] = np.average(pc_coords.T[angle_indices],
                                 axis=0,
                                 weights=flux[angle_indices])
        pc_stds[:, i] = np.sqrt(np.average((pc_coords.T[angle_indices] - pc_means[:, i])**2,
                                         axis=0,
                                         weights=flux[angle_indices]))
        
    
    return pc_coords, pc_means, pc_stds, partitions


# Extract 1D streamline from the data cube - original method
pc_coords, pc_means, pc_stds, partitions = reduce_to_1D_polar(streamer_cube, n_elements=n_points)
print(f"point cloud velocities (km/s): {pc_coords[2]}")


In [ ]:
# Prepare data for gradient descent
ra_data = pc_means[0] # offsets in arcsec
dec_data = pc_means[1] # offsets in arcsec
v_data = pc_means[2]   # velocities in km/s (rel to vlsr)
print(f"data velocities (km/s): {v_data}")

ra_sigma = pc_stds[0]
dec_sigma = pc_stds[1]
v_sigma = pc_stds[2]


data = (ra_data, dec_data, v_data)
uncertainties = (ra_sigma, dec_sigma, v_sigma)

### !Monkey-patch of matching metric for angular bins!
Local, does not change source code

In [ ]:
# Monkey-patch the matching metric so fit_streamline matches in angular space instead of radial space.
# This stays local to the notebook session and does not modify the package source.
_original_get_distance_metric = extract_streamline.get_distance_metric

_theta_reference = float(np.median(partitions))

def get_angular_distance_metric(ra_coords, dec_coords, return_trace=False):
    _r_proj, theta_proj = extract_streamline.cartesian_to_polar(
        jnp.asarray(ra_coords, dtype=jnp.float64),
        jnp.asarray(dec_coords, dtype=jnp.float64),
    )
    theta_metric = extract_streamline._wrap_to_pi(theta_proj - _theta_reference)

    if return_trace:
        trace = {
            'n_points': int(jnp.asarray(theta_metric).size),
            'theta_reference': float(_theta_reference),
        }
        return theta_metric, trace

    return theta_metric

extract_streamline.get_distance_metric = get_angular_distance_metric
print(f"Patched matching metric to angular bins using theta_ref={_theta_reference:.6f} rad")


In [ ]:
def plot_angle_bin_lines(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5):
    """
    Plot rays showing the polar angle bin edges on an RA-Dec plot.
    Axis limits are preserved after adding the lines.
    
    Parameters
    ----------
    ax : matplotlib axes object
        The axes to plot on
    partitions : array
        Polar angle bin boundaries in radians
    color : str
        Color of the lines
    linewidth : float
        Line width of the lines
    alpha : float
        Transparency of the lines
    """
    import numpy as np
    # Save current axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    # Add rays that start at the star and extend only in the forward angular direction
    x_min, x_max = sorted(xlim)
    y_min, y_max = sorted(ylim)
    for partition in partitions:
        dx = np.cos(partition)
        dy = np.sin(partition)
        ray_lengths = []
        if dx != 0:
            for x_edge in (x_min, x_max):
                t = x_edge / dx
                if t > 0:
                    y_at_edge = t * dy
                    if y_min <= y_at_edge <= y_max:
                        ray_lengths.append(t)
        if dy != 0:
            for y_edge in (y_min, y_max):
                t = y_edge / dy
                if t > 0:
                    x_at_edge = t * dx
                    if x_min <= x_at_edge <= x_max:
                        ray_lengths.append(t)
        if not ray_lengths:
            continue
        t_max = min(ray_lengths)
        x_vals = np.array([0.0, t_max * dx])
        y_vals = np.array([0.0, t_max * dy])
        ax.plot(x_vals, y_vals, color=color, linewidth=linewidth, alpha=alpha)
    
    # Restore original axis limits
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

In [ ]:
# plot it
import matplotlib.pyplot as plt
# plot the observed data points as a scatter
plt.scatter(pc_coords[0], pc_coords[1], s=1, alpha=0.3, color='grey', label='Point cloud')
# plot the extracted 1D streamline with error bars
plt.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', label='Extracted 1D Streamline', color='red')
# plot the star
plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
# plot the angular bin edges
ax = plt.gca()
plot_angle_bin_lines(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5)
plt.xlabel('RA Offset (arcsec)')
plt.ylabel('Dec Offset (arcsec)')
# flip the x axis to match the astronomical convention (RA increases to the left)
plt.gca().invert_xaxis()
plt.legend()
plt.title('Extracted 1D Streamer emission')


### Initial guess at parameters, and plot

This is so you can refine your initial parameters a bit. e.g. get the projected radius about right

In [ ]:
# Parameters to optimize
initial_opt_params = {
    'r0': 2000.0,  # au
    'theta0': 80.0,  # degrees
    'phi0': 90.0,  # degrees
    'log_omega': np.log(3e-13),  # log(1/s)
    'v_r0': 0.1,  # km/s
}

# B5IRS1
fixed_params = {
    'mass': 0.2,  # solar masses
    'inc': 13.0,  # degrees
    'pa': (157.1+90),  # degrees
    'rmin': 20.0,  # au
    'deltar':10.0,  # au
    'v_lsr': 10.2  # km/s (systemic velocity)
}

# Convert angles from degrees to radians
initial_opt_params['theta0'] = np.radians(initial_opt_params['theta0'])
initial_opt_params['phi0'] = np.radians(initial_opt_params['phi0'])
fixed_params['inc'] = np.radians(fixed_params['inc'])
fixed_params['pa'] = np.radians(fixed_params['pa'])

# generate the initial guess model
ra_guess, dec_guess, v_guess = gradient_descent.forward_model(initial_opt_params, fixed_params, distance)

# plot it on top of the data and the extracted 1D streamline
# plot the observed data points as a scatter
plt.scatter(pc_coords[0], pc_coords[1], s=1, alpha=0.3, color='grey', label='Point cloud')
# plot the extracted 1D streamline with error bars
plt.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', label='Extracted 1D Streamline', color='red')
# plot the initial guess streamer
plt.plot(ra_guess, dec_guess, color='blue', label='Streamline from stream_lines_grad')
# plot the angular bin edges
ax = plt.gca()
plot_angle_bin_lines(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5)
# plot the star
plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
plt.xlabel('RA Offset (arcsec)')
plt.ylabel('Dec Offset (arcsec)')
# flip the x axis to match the astronomical convention (RA increases to the left)
plt.gca().invert_xaxis()
plt.legend()
plt.title('Extracted 1D Streamer emission')




### TEST Forward model and calculating loss (with extra plots)

First set your input params in the format required

Doing chi2 loss more manually, just so we can check the model point choosing is working properly (delete this later)

## Full fit streamline starts here

In [ ]:
def get_omega(mass, r0):
    '''
    this gets value of omega when r_cent = 0.5 * r0
    '''
    omega_squared = 0.5 * G * mass / (jnp.power(r0, 3) * jnp.power(au_in_km, 2)) # in s^-2
    omega = jnp.power(omega_squared, 0.5) # in s^-1
    return omega

opt_params = initial_opt_params.copy()

# Define physically reasonable bounds (omega bounds transformed to natural log space)
# These bounds are also used as normalization anchors: x_norm = (x - min) / (max - min).
# Provide bounds for every optimized parameter.
r0_min, r0_max = 200.0, 20000.0 # param bounds in au

# the omega bounds are set by keeping centrifugal radius reasonable (r_cent = 0.5 r0)
omega_max = get_omega(fixed_params['mass'], r0_min)
omega_min = get_omega(fixed_params['mass'], r0_max)
# print these in scientific notation for sanity check
print(f"Omega bounds: {omega_min:.2e} to {omega_max:.2e} 1/s")

param_bounds = {
    'r0': (r0_min, r0_max),                    # radius between 200-20000 au
    'theta0': (0.0, np.pi),                    # polar angle 0-pi
    'phi0': (0.0, 2*np.pi),                    # azimuthal angle 0-2pi
    'log_omega': (np.log(omega_min), np.log(omega_max)),  # omega in [omega_min, omega_max] 1/s
    'v_r0': (-5.0, 5.0),                       # radial velocity -5 to 5 km/s
}

log_file = 'streamfit_test_output/optimisation_log.csv'
trace_file = 'streamfit_test_output/optimisation_trace.csv'
trace_every = 1
n_epochs = 300
info_every = 10
learning_rate = 0.005 # Single learning rate applied to all normalized optimization parameters
loss_method = 'rthetavel'

gradient_tol = 1e-2 * len(initial_opt_params) # gradient tolerance scaled by number of parameters

## here we run the fit, using cProfile to track performance
profile = False # set to True to enable cProfile profiling of the optimization run
if profile:
    profiler = cProfile.Profile()
    profiler.enable()

best_opt_params, loss_history, param_errors = gradient_descent.fit_streamline(
    opt_params,
    fixed_params,
    data,
    uncertainties,
    distance,
    learning_rate=learning_rate,
    param_bounds=param_bounds,
    n_epochs=n_epochs,
    info_every=info_every,
    loss_threshold=0.05,
    loss_threshold_epochs=5,
    gradient_tol=gradient_tol,
    gradient_tol_epochs=5,
    early_stopping_patience=80,
    log_file=log_file,
    trace_file=trace_file,
    trace_every=trace_every,
    loss_method=loss_method,
    output_uncertainties=True,
 )

if profile:
    profiler.disable() # Stop profiling after optimization is complete


print(f"Optimized using loss_method='{loss_method}'")

# Plot loss history
# Epoch indexing: epoch 0 = initial state, epoch i (i >= 1) = after update i
# loss_history is 0-indexed: loss_history[i] = loss at epoch i
plt.figure(figsize=(8, 5))
epochs = range(len(loss_history))
plt.plot(epochs, loss_history, marker='o', markersize=4)
plt.xlabel('Epoch (0 = initial, i = after update i)')
plt.ylabel('Loss')
plt.title('Optimization Progress\n(Epoch i = loss after applying i updates; epoch 0 = initial)')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:

if profile:
    # Analyze profiling results
    # to see it with snakeviz: snakeviz streamfit_runtime_profile.prof in terminal
    print("\ncProfiling results: -----------------------")
    stats = pstats.Stats(profiler)
    stats.sort_stats('cumulative')
    stats.print_stats(20)  # Print top 20 functions by cumulative time
    stats.dump_stats('streamfit_runtime_profile.prof')
    print("------------------------\n")

## Best fit visualisation

In [ ]:
# Final model with best-fit parameters
ra_best, dec_best, v_best = gradient_descent.forward_model(best_opt_params, fixed_params, distance)

# remove NaN values (due to rmin) from model for plotting
not_nan = ~jnp.isnan(ra_best) & ~jnp.isnan(dec_best) & ~jnp.isnan(v_best)
ra_best = ra_best[not_nan]
dec_best = dec_best[not_nan]
v_best = v_best[not_nan]

# plot it on top of the data
plt.scatter(pc_coords[0], pc_coords[1], s=1, color='gray', alpha=0.3, label='Point cloud', zorder=4)
plt.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', label='Extracted 1D Streamline', color='red')
plt.plot(ra_best, dec_best, color='blue', linewidth=2, label='Best-fit Model Streamline')

# get the model positions at data arc length points (overlap restricted)
ra_best_interp, dec_best_interp, v_best_interp, valid, _dmetric_model, _overlap_min, _overlap_max = gradient_descent.match_model_to_data_curve(
    ra_best, dec_best, v_best, ra_data, dec_data)

# plot model positions only where overlap is retained
plt.scatter(
    ra_best_interp[valid], dec_best_interp[valid],
    s=25, label='Model at retained data arc lengths', color='blue', zorder=5
)
plt.scatter(
    ra_data[valid], dec_data[valid],
    s=45, facecolor='none', edgecolor='cyan', linewidth=1.2,
    label='Retained data points', zorder=6
)

# plot the angular bin edges
ax = plt.gca()
plot_angle_bin_lines(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5)

plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
plt.xlabel('RA Offset (arcsec)')
plt.ylabel('Dec Offset (arcsec)')

# Save axis limits before adding background/circles
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()


# Restore original axis limits
ax.set_xlim(xlim)
ax.set_ylim(ylim)

plt.gca().invert_xaxis()
plt.legend()
plt.title('Best-fit Streamline Model')
plt.show()

# also plot the same thing in velocity vs projected radius space


## Uncertainty visualisations

In [ ]:
# Uncertainty visualizations: diagonal error bars, parameter correlation, and streamline spaghetti
import numpy as np
import matplotlib.pyplot as plt

# Use only optimized parameters (exclude derived omega from best_opt_params if present)
opt_keys = list(initial_opt_params.keys())
best_for_cov = {k: float(best_opt_params[k]) for k in opt_keys}

# Prepare data-only quantities once
prepared_data = extract_streamline.prepare_data(data, uncertainties)

# Recover covariance from Hessian-based uncertainty estimate
param_errors_cov, cov = gradient_descent.estimate_parameter_errors(
    best_for_cov,
    fixed_params,
    data,
    uncertainties,
    distance,
    prepared_data,
    loss_method=loss_method,
    gradient_tol=gradient_tol,
    normalization_spec=None,
    )

param_errors_plot = {k: float(param_errors[k]) for k in opt_keys}

# ---------- 1) Normalized parameter error bars ----------
param_vals = np.array([best_for_cov[k] for k in opt_keys], dtype=float)
param_errs = np.array([param_errors_plot[k] for k in opt_keys], dtype=float)

# Avoid divide-by-zero issues
eps = 1e-12

# Relative (fractional) errors
norm_errs = param_errs / (np.abs(param_vals) + eps)

fig, ax = plt.subplots(figsize=(8, 4.5))

ypos = np.arange(len(opt_keys))

ax.barh(
    ypos,
    norm_errs,
    color='tab:blue',
    alpha=0.8
)

ax.set_yticks(ypos)
ax.set_yticklabels(opt_keys)

ax.set_xlabel('Relative uncertainty ($\\sigma / |x|$)')
ax.set_title('Normalized Parameter Uncertainties')

ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

# ---------- 2) Correlation heatmap from covariance ----------
# normalised correlation_{i,j} = covariance_{i,j} / (sigma_i * sigma_j)
cov_np = np.array(cov, dtype=float)
print("Covariance matrix:")
print(cov_np)
diag = np.sqrt(np.clip(np.diag(cov_np), 1e-30, None))
corr = cov_np / np.outer(diag, diag)
corr = np.clip(corr, -1.0, 1.0)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(np.arange(len(opt_keys)))
ax.set_yticks(np.arange(len(opt_keys)))
ax.set_xticklabels(opt_keys, rotation=45, ha='right')
ax.set_yticklabels(opt_keys)
ax.set_title('Parameter Correlation Matrix')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Correlation coefficient')

for i in range(len(opt_keys)):
    for j in range(len(opt_keys)):
        ax.text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center', fontsize=8, color='black')

plt.tight_layout()
plt.show()

# ---------- 3) Streamline spaghetti from covariance sampling ----------
rng = np.random.default_rng(7)
mu = np.array([best_for_cov[k] for k in opt_keys], dtype=float)

n_samples = 150
samples = rng.multivariate_normal(mu, cov_np, size=n_samples)

# Clipping to user bounds if present
for j, key in enumerate(opt_keys):
    if key in param_bounds:
        lo, hi = param_bounds[key]
        samples[:, j] = np.clip(samples[:, j], lo, hi)

fig, (ax_sky, ax_v) = plt.subplots(1, 2, figsize=(14, 5))

# Plot sampled streamlines
for s in samples:
    sample_params = {k: float(v) for k, v in zip(opt_keys, s)}
    try:
        ra_s, dec_s, v_s = gradient_descent.forward_model(sample_params, fixed_params, distance)
        ra_s = np.array(ra_s, dtype=float)
        dec_s = np.array(dec_s, dtype=float)
        v_s = np.array(v_s, dtype=float)
        finite = np.isfinite(ra_s) & np.isfinite(dec_s) & np.isfinite(v_s)
        if np.sum(finite) < 3:
            continue

        ra_f = ra_s[finite]
        dec_f = dec_s[finite]
        v_f = v_s[finite]
        d_f = np.array(extract_streamline.get_distance_metric(ra_f, dec_f), dtype=float)
        ord_idx = np.argsort(d_f)

        ax_sky.plot(ra_f, dec_f, color='tab:blue', alpha=0.06, lw=1)
        ax_v.plot(d_f[ord_idx], v_f[ord_idx], color='tab:blue', alpha=0.06, lw=1)
    except Exception:
        # Skip pathological sampled parameter combinations
        continue

# Overlay best-fit streamline
ra_best_plot = np.array(ra_best, dtype=float)
dec_best_plot = np.array(dec_best, dtype=float)
v_best_plot = np.array(v_best, dtype=float)
d_best = np.array(extract_streamline.get_distance_metric(ra_best_plot, dec_best_plot), dtype=float)
ord_best = np.argsort(d_best)

ax_sky.plot(ra_best_plot, dec_best_plot, color='blue', lw=2, label='Best-fit')
ax_sky.errorbar(
    ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma,
    fmt='o', color='red', ecolor='red', ms=4, alpha=0.9, label='Data'
    )
ax_sky.invert_xaxis()
ax_sky.set_xlabel('RA Offset (arcsec)')
ax_sky.set_ylabel('Dec Offset (arcsec)')
ax_sky.set_title('Sky-plane Streamline Spaghetti')
ax_sky.legend()

ax_v.plot(d_best[ord_best], v_best_plot[ord_best], color='blue', lw=2, label='Best-fit')
d_data = np.array(extract_streamline.get_distance_metric(ra_data, dec_data), dtype=float)
ord_data = np.argsort(d_data)
ax_v.errorbar(
    d_data[ord_data], np.array(v_data)[ord_data], yerr=np.array(v_sigma)[ord_data],
    fmt='o', color='red', ecolor='red', ms=4, alpha=0.9, label='Data'
    )
ax_v.set_xlabel('Projected distance (arcsec)')
ax_v.set_ylabel('Velocity (km/s)')
ax_v.set_title('Velocity Spaghetti')
ax_v.legend()

plt.tight_layout()
plt.show()